# Bilinear rank sweep - is rank 64 a lucky number, or is the effect robust?

**What this measures.** The `bilinear` fusion block multiplies each pair of views together
through a low-rank projection. "Rank" sets how big that interaction is allowed to be: the
full version would need a 256x256 matrix per pair, and the low-rank form replaces it with
two 256 x r projections. Every result in this project so far used **r = 64**, chosen once
and never questioned.

Section 5.6 of the draft made that choice matter. Across both training settings, the
bilinear term is the part of the proposed fusion that actually does work, and
cross-attention is the part that does not. So the obvious question a reviewer asks is
whether r = 64 was tuned into looking good. This run answers it by measuring three other
ranks under exactly the same protocol.

| rank | parameters in the fusion block | vs r=64 |
|---|---|---|
| 16 | 24,768 | 0.25x |
| 32 | 49,536 | 0.5x |
| **64 (already measured)** | **99,072** | **1x** |
| 128 | 198,144 | 2x |

An 8x span. If the bilinear effect is real, the three new ranks should land near r = 64 or
trend smoothly with rank. If r = 64 sits alone above a scatter of the others, it was luck,
and section 5.6 needs to say so.

**Why r = 64 is not re-run here.** It already exists, trained on a Colab T4, as the archived
tag `fuse_bilinear_gpu`. Re-running it would burn a quarter of this session's GPU time to
reproduce a number we have. More importantly, comparing the new ranks against the *CPU*
`fuse_bilinear` would be a cross-device comparison, and this project measured what that
costs: 18% of single-split numbers move by more than the minimum detectable effect from
nothing but a change of hardware. So the sweep runs on a T4 and is compared against the T4
r = 64 run. Same device, same code, same splits.

**Where the results land.** Everything archives under `_gpu`-suffixed tags
(`fuse_bilinear_r16_gpu` and so on), so nothing can overwrite an existing result and the
device is visible in the tag as well as in the metrics rows.

Expected: **1.5-2.5 h on a T4** for three tags across six splits. `--resume` continues after
a timeout, so re-run the training cell as many times as it takes.

## 1. Check you actually got a GPU

If this says `cuda: False`, the point of the run is lost - it would train on a CPU, and the
results would not be comparable to `fuse_bilinear_gpu`.

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('No GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.')

## 2. Connect your Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Point at the bundle

Upload **`mpp_rank_sweep.zip`** (132 MB) to Drive first. It carries the cached ChemBERTa
embeddings, which is what lets `--seq cached` run without loading the transformer.

This is a *fresh* bundle, not the Phase 3 one - `src/train/loop.py` gained the `device` and
`seed` columns after that bundle was built, and this run should record its own provenance
rather than leave it to be inferred later.

In [ ]:
BUNDLE = '/content/drive/MyDrive/mpp_rank_sweep.zip'  # edit if elsewhere
OUTDIR = '/content/drive/MyDrive/mpp_rank_sweep'

import os
assert os.path.exists(BUNDLE), f'Not found: {BUNDLE} -- check path, re-run.'
os.makedirs(OUTDIR, exist_ok=True)
print('bundle:', round(os.path.getsize(BUNDLE)/1e6, 1), 'MB')

## 4. Unpack and install

`torchao` is uninstalled deliberately: an old build's probe *raises* instead of returning
False, which kills every run that touches `peft`.

In [ ]:
import zipfile, os

WORK = '/content/mpp'
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(WORK)
os.chdir(WORK)

!pip -q install peft accelerate torch_geometric
!pip -q uninstall -y torchao

import torch_geometric
print('torch_geometric', torch_geometric.__version__)

## 5. Keep results on Drive

`results/runs` becomes a symlink into Drive, so a disconnect loses nothing and `--resume`
can see what already finished. This is its own folder, so it cannot collide with any other
run's archive.

In [ ]:
import os

os.makedirs(f'{OUTDIR}/runs', exist_ok=True)
os.makedirs('results', exist_ok=True)
if not os.path.islink('results/runs'):
    if os.path.exists('results/runs'):
        import shutil; shutil.rmtree('results/runs')
    os.symlink(f'{OUTDIR}/runs', 'results/runs')
print('results/runs ->', os.path.realpath('results/runs'))

## 6. Smoke test - do not skip this

One epoch on the smallest dataset, at the largest rank and then the smallest. If `--rank`
is not plumbed through to the fusion block on this runtime, it fails here in a minute
rather than two hours in.

**Read the two parameter counts.** They must differ by exactly **259,392** - that is
173,376 in the bilinear block itself, plus 86,016 in the head that reads it (the block's
output is 384-wide at r=128 against 48-wide at r=16, and the head's first layer is 256
units either way). If the two counts are equal, `--rank` is being ignored and the sweep
would silently measure the same model three times.

Measured locally: r=128 reports 2,265,093 params on FreeSolv and r=16 reports 2,005,701.
Those exact totals depend on the number of tasks, so on another dataset they shift together
- but the *difference* is driven only by the rank, so it holds everywhere.

The smoke outputs are deleted immediately so they cannot be mistaken for results.

In [ ]:
!python -m src.data.materialize --variant deepchem --artifacts chemberta ecfp graphs desc
!python -m src.train.train_fusion --mode bilinear --tag _smoke_r128 --seq cached --rank 128 --datasets freesolv --epochs 1 --patience 1 --device cuda
!python -m src.train.train_fusion --mode bilinear --tag _smoke_r16 --seq cached --rank 16 --datasets freesolv --epochs 1 --patience 1 --device cuda
!rm -f results/metrics/*_smoke_r*.csv results/preds/*_smoke_r*.npy
!rm -f models/*_smoke_r*.pt
print('smoke test done and cleaned up -- the two param counts above must differ by 259,392')

## 7. Train

`--seq cached` is what makes this the *frozen* setting, matching `fuse_bilinear_gpu` rather
than the end-to-end ladder. `--tag-suffix gpu` keeps the results in their own namespace:
they archive as `fuse_bilinear_r16_gpu` and cannot overwrite anything.

`--resume` skips whatever already finished, so re-running this cell after a timeout
continues rather than restarting.

In [ ]:
!python -m scripts.run_view_multiseed --tags fuse_bilinear_r16 fuse_bilinear_r32 fuse_bilinear_r128 --variants deepchem seed0 seed1 seed2 seed3 seed4 --artifacts chemberta ecfp graphs desc --seq cached --device cuda --tag-suffix gpu --resume --restore none

## 8. Check what finished

16 metrics and 16 predictions per tag per split: 8 datasets x (valid, test).

In [ ]:
import glob
tags = ['fuse_bilinear_r16_gpu', 'fuse_bilinear_r32_gpu', 'fuse_bilinear_r128_gpu']
allok = True
for v in ['deepchem','seed0','seed1','seed2','seed3','seed4']:
    m = [len(glob.glob(f'{OUTDIR}/runs/{v}/metrics/*_{t}_*.csv')) for t in tags]
    p = [len(glob.glob(f'{OUTDIR}/runs/{v}/preds/*_{t}_*.npy')) for t in tags]
    done = all(c == 16 for c in m) and all(c == 16 for c in p)
    allok &= done
    print(v, 'metrics', dict(zip(tags, m)), 'preds', dict(zip(tags, p)),
          'ok' if done else 'INCOMPLETE - re-run the training cell')
print()
print('ALL DONE - run the last cell' if allok else 'Not finished yet. Re-run the training cell.')

## 9. Sanity check: did every split really train on the GPU?

Reads one metrics row per tag. If any row says `cpu`, that split trained on the wrong
hardware and is not comparable to `fuse_bilinear_gpu` - delete it from Drive and re-run the
training cell with a GPU runtime.

In [ ]:
import glob, pandas as pd
for t in ['fuse_bilinear_r16_gpu', 'fuse_bilinear_r32_gpu', 'fuse_bilinear_r128_gpu']:
    files = sorted(glob.glob(f'{OUTDIR}/runs/*/metrics/*_{t}_test.csv'))
    if not files:
        print(f'{t:<28} nothing yet')
        continue
    devs = {str(pd.read_csv(f).iloc[0].get('device', 'NOT RECORDED')) for f in files}
    print(f'{t:<28} {len(files):>3} test rows, device(s): {sorted(devs)}')

## 10. Bring the results home

Then, locally and **with `--dry-run` first**:

```
python -m scripts.merge_colab_results ~/Downloads/rank_sweep_results.zip --dry-run
```

Unzipping by hand nests `results/{metrics,preds,figs,logs}` inside `results/runs/`, which
reads as a mass deletion. The merge script exists to prevent exactly that.

In [ ]:
import shutil, os
out = shutil.make_archive('/content/rank_sweep_results', 'zip', f'{OUTDIR}/runs')
print('wrote', out, round(os.path.getsize(out)/1e6, 2), 'MB')
from google.colab import files
files.download(out)